In [164]:
import numpy as np
from typing import List, Tuple, Dict

## Câu hỏi tự luận

In [165]:
def create_training_data() -> np.ndarray:
    """Create the training dataset for tennis prediction."""
    data = [
    ["Sunny", "Hot", "High", "Weak", "No"],
    ["Sunny", "Hot", "High", "Strong", "No"],
    ["Overcast", "Hot", "High", "Weak", "Yes"],
    ["Rain", "Mild", "High", "Weak", "Yes"],
    ["Rain", "Cool", "Normal", "Weak", "Yes"],
    ["Rain", "Cool", "Normal", "Strong", "No"],
    ["Overcast", "Cool", "Normal", "Strong", "Yes"],
    ["Overcast", "Mild", "High", "Weak", "No"],
    ["Sunny", "Cool", "Normal", "Weak", "Yes"],
    ["Rain", "Mild", "Normal", "Weak", "Yes"]
    ]
    return np.array(data)

train_data = create_training_data()
print(train_data)

[['Sunny' 'Hot' 'High' 'Weak' 'No']
 ['Sunny' 'Hot' 'High' 'Strong' 'No']
 ['Overcast' 'Hot' 'High' 'Weak' 'Yes']
 ['Rain' 'Mild' 'High' 'Weak' 'Yes']
 ['Rain' 'Cool' 'Normal' 'Weak' 'Yes']
 ['Rain' 'Cool' 'Normal' 'Strong' 'No']
 ['Overcast' 'Cool' 'Normal' 'Strong' 'Yes']
 ['Overcast' 'Mild' 'High' 'Weak' 'No']
 ['Sunny' 'Cool' 'Normal' 'Weak' 'Yes']
 ['Rain' 'Mild' 'Normal' 'Weak' 'Yes']]


In [166]:
def compute_prior_probabilities(train_data: np.ndarray) -> np.ndarray:
    """
    Calculate prior probabilities P(Play Tennis = Yes/No).

    Args:
        train_data: Training dataset

    Returns:
        Array of prior probabilities [P(No), P(Yes)]
    """
    class_names = ['No', 'Yes']
    total_samples = len(train_data)
    prior_probs = np.zeros(len(class_names))

    # Tính số lần xuất hiện yes/no
    for i in train_data[:,-1]:
      if i == "No":
        prior_probs[0] += 1
      else:
        prior_probs[1] += 1
    # Calculate probs
    prior_probs = prior_probs/ total_samples

    return prior_probs

prior_probablity = compute_prior_probabilities(train_data)
print("P(“Play Tennis” = No)", prior_probablity[0])
print("P(“Play Tennis” = Yes)", prior_probablity[1])

P(“Play Tennis” = No) 0.4
P(“Play Tennis” = Yes) 0.6


In [167]:
def compute_conditional_probabilities(train_data: np.ndarray) -> tuple:
    """
    Calculate conditional probabilities P(Feature|Class) for all features.

    Args:
        train_data: Training dataset

    Returns:
        Tuple of (conditional_probabilities, feature_values)
    """
    class_names = ['No', 'Yes']
    n_features = train_data.shape[1] - 1  # Exclude target column
    conditional_probs = []
    feature_values = []

    for feature_idx in range(n_features):
        # Get unique values for this feature -> Theo từng cột, Example: cột thứ 3: Weather -> unique value: sunny, rain,...
        unique_values = np.unique(train_data[:, feature_idx])
        feature_values.append(unique_values)

        # Initialize conditional probability matrix
        feature_cond_probs = np.zeros((len(class_names), len(unique_values)))

        for class_idx, class_name in enumerate(class_names):
            # Get samples for this class
            samples = train_data[train_data[:,-1] == class_name]   # Lấy list những sample cùng No/Yes

            for value_idx, value in enumerate(unique_values):
                # Count occurrences of this feature value in this class
                sub_samples = samples[:,feature_idx] # Lấy cột của feature đang tính
                count = 0
                for sample in sub_samples:
                  if sample == value:
                    count += 1
                # Calculate conditional probability
                feature_cond_probs[class_idx][value_idx]  = count/len(samples)
                ### Your code here

        conditional_probs.append(feature_cond_probs)

    return conditional_probs, feature_values

In [168]:
_, feature_values  = compute_conditional_probabilities(train_data)
print("x1 = ",feature_values[0])
print("x2 = ",feature_values[1])
print("x3 = ",feature_values[2])
print("x4 = ",feature_values[3])

x1 =  ['Overcast' 'Rain' 'Sunny']
x2 =  ['Cool' 'Hot' 'Mild']
x3 =  ['High' 'Normal']
x4 =  ['Strong' 'Weak']


In [169]:
def train_naive_bayes(train_data):
    """
    Train the Naive Bayes classifier.

    Args:
        train_data: Training dataset

    Returns:
        Tuple of (prior_probabilities, conditional_probabilities, feature_names)
    """

    # Calculate prior probabilities
    prior_probabilities = compute_prior_probabilities(train_data)

    # Calculate conditional probabilities
    conditional_probabilities, feature_names = compute_conditional_probabilities(train_data)

    return prior_probabilities, conditional_probabilities, feature_names

In [170]:
# Train the model
prior_probs, conditional_probs, feature_names = train_naive_bayes(train_data)

In [171]:
def get_feature_index(feature_value, feature_values):
    """
    Get the index of a feature value in the feature values array.

    Args:
        feature_value: Value to find
        feature_values: Array of possible feature values

    Returns:
        Index of the feature value
    """
    return np.argmax(feature_values == feature_value)

In [172]:
outlook = feature_values[0]
i1 = get_feature_index("Overcast", outlook)
i2 = get_feature_index("Rain", outlook)
i3 = get_feature_index("Sunny", outlook)

print(i1, i2, i3)

0 1 2


In [173]:
# Compute P("Outlook"="Sunny"|Play Tennis"="Yes")
x1 = get_feature_index("Sunny",feature_values[0])
print("P('Outlook'='Sunny'|Play Tennis'='Yes') = ",
      np.round(conditional_probs[0][1, x1],2)
)

P('Outlook'='Sunny'|Play Tennis'='Yes') =  0.17


In [174]:
# Compute P("Outlook"="Sunny"|Play Tennis"="No")
x1 = get_feature_index("Sunny",feature_values[0])
print(
    "P('Outlook'='Sunny'|Play Tennis'='No') = ",
    np.round(conditional_probs[0][0, x1],2)
)

P('Outlook'='Sunny'|Play Tennis'='No') =  0.5


In [175]:
def predict_tennis(
        X, prior_probabilities, conditional_probabilities, feature_names
    ):
    """
    Make a prediction for given features.

    Args:
        X: List of feature values [Outlook, Temperature, Humidity, Wind]
        prior_probabilities: Prior probabilities for each class
        conditional_probabilities: Conditional probabilities for each feature
        feature_names: Names/values for each feature

    Returns:
        Tuple of (prediction, probabilities)
    """
    class_names = ['No', 'Yes']

    # Get feature indices
    feature_indices = []
    for i, feature_value in enumerate(X):
        feature_indices.append(get_feature_index(feature_value, feature_names[i]))

    # Calculate probabilities for each class
    class_probabilities = []

    for class_idx in range(len(class_names)):
        # Start with prior probability
        prior_prob = prior_probabilities[class_idx]

        class_probabilities.append(prior_prob)
        # Multiply by conditional probabilities
        for i in range(len(conditional_probabilities)):
          conditional_prob = conditional_probabilities[i][class_idx][feature_indices[i]]
          class_probabilities[class_idx] *= conditional_prob

    # Normalize probabilities
    total_prob = sum(class_probabilities)
    if total_prob > 0:
        normalized_probs = [p / total_prob for p in class_probabilities]
    else:
        normalized_probs = [0.5, 0.5]  # Default if all probabilities are 0

    # Make prediction
    predicted_class_idx = np.argmax(class_probabilities)
    prediction = class_names[predicted_class_idx]

    # Create probability dictionary
    prob_dict = {
        'No': round(normalized_probs[0].item(), 2),
        'Yes': round(normalized_probs[1].item(), 2)
    }

    return prediction, prob_dict

In [176]:
X = ['Sunny','Cool', 'High', 'Strong']
prior_probs, conditional_probs, feature_names = train_naive_bayes(train_data)
prediction, prob_dict = predict_tennis(
    X, prior_probs, conditional_probs, feature_names
)
if prediction:
    print("Ad should go!")
else:
    print("Ad should not go!")
prediction, prob_dict

Ad should go!


('No', {'No': 0.87, 'Yes': 0.13})

## Câu hỏi trắc nghiệm

### III.1. Binary Classification - Play Tennis

In [177]:
# Câu 1: Xác suất xảy ra sự kiện "Play Tennis"="Yes" và "Play Tennis"="No" lần lượt là?
prior_probablity = compute_prior_probabilities(train_data)
print("P(“Play Tennis” = No)", prior_probablity[0])
print("P(“Play Tennis” = Yes)", prior_probablity[1])

# -> Câu A

P(“Play Tennis” = No) 0.4
P(“Play Tennis” = Yes) 0.6


In [178]:
# Câu 2: Xác suất xảy ra sự kiện "Play Tennis"="Yes" khi sự kiện X xảy ra là?
X = ['Sunny','Cool', 'High', 'Strong']
prior_probs, conditional_probs, feature_names = train_naive_bayes(train_data)

class_names = ['No', 'Yes']

# Get feature indices
feature_indices = []
for i, feature_value in enumerate(X):
    feature_indices.append(get_feature_index(feature_value, feature_names[i]))

# Calculate probabilities
class_probability = prior_probs[1]

# Start with prior probability
prior_prob = prior_probs[1]

# Multiply by conditional probabilities
for i in range(len(conditional_probs)):
  conditional_prob = conditional_probs[i][1][feature_indices[i]]
  class_probability *= conditional_prob
class_probability

# Câu B - xấp xỉ 0.0028

np.float64(0.002777777777777777)

In [179]:
# Câu 3. Xác suất xảy ra sự kiện "Play Tennis"="No" khi sự kiện X xảy ra là?

X = ['Sunny','Cool', 'High', 'Strong']
prior_probs, conditional_probs, feature_names = train_naive_bayes(train_data)

class_names = ['No', 'Yes']

# Get feature indices
feature_indices = []
for i, feature_value in enumerate(X):
    feature_indices.append(get_feature_index(feature_value, feature_names[i]))

# Calculate probabilities
class_probability = prior_probs[0]

# Start with prior probability
prior_prob = prior_probs[0]

# Multiply by conditional probabilities
for i in range(len(conditional_probs)):
  conditional_prob = conditional_probs[i][0][feature_indices[i]]
  class_probability *= conditional_prob
class_probability

# Câu C: 0.0188

np.float64(0.018750000000000003)

In [180]:
# Câu 4: B "No"

In [181]:
# Câu 5: A

In [182]:
# Câu 6: B

In [183]:
# Câu 7: C

In [184]:
# Câu 8 B

### III.2. Multi-label Classification - Traffic Data

In [194]:
traffic_data =  [
    ["Weekday", "Spring", "None", "None", "On Time"],
    ["Weekday", "Winter", "None", "Slight", "On Time"],
    ["Weekday", "Winter", "None", "None", "On Time"],
    ["Holiday", "Winter", "High", "Slight", "Late"],
    ["Saturday", "Summer", "Normal", "None", "On Time"],
    ["Weekday", "Autumn", "Normal", "None", "Very Late"],
    ["Holiday", "Summer", "High", "Slight", "On Time"],
    ["Sunday", "Summer", "Normal", "None", "On Time"],
    ["Weekday", "Winter", "High", "Heavy", "Very Late"],
    ["Weekday", "Summer", "None", "Slight", "On Time"],
    ["Saturday", "Spring", "High", "Heavy", "Cancelled"],
    ["Weekday", "Summer", "High", "Slight", "On Time"],
    ["Weekday", "Winter", "Normal", "None", "Late"],
    ["Weekday", "Summer", "High", "None", "On Time"],
    ["Weekday", "Winter", "Normal", "Heavy", "Very Late"],
    ["Saturday", "Autumn", "High", "Slight", "On Time"],
    ["Weekday", "Autumn", "None", "Heavy", "On Time"],
    ["Holiday", "Spring", "Normal", "Slight", "On Time"],
    ["Weekday", "Spring", "Normal", "None", "On Time"],
    ["Weekday", "Spring", "Normal", "Heavy", "On Time"]
]

In [186]:
train_data_traffic = np.array(traffic_data)

In [187]:
X = ["Weekday", "Winter", "High", "Heavy"]

In [188]:
# Câu 9 Xác suất xảy ra sự kiện "Class" = "On Time", sự kiện "Class" = "Late", sự kiện "Class" = "Very Late" và sự kiện "Class" = "Cancelled" lần lượt là?

def compute_prior_probabilities_traffic(train_data: np.ndarray) -> np.ndarray:
    """
    Calculate prior probabilities P(Class = On Time/Late/ Very Late/ Cancelled).

    Args:
        train_data: Training dataset

    Returns:
        Array of prior probabilities [On Time, Late, Very Late, Cancelled]
    """
    class_dict = {}
    total_samples = len(train_data)
    prior_probs = np.zeros(len(class_names))

    # Tính số lần xuất hiện yes/no
    for traffic_class in train_data[:,-1]:
      if traffic_class in class_dict.keys():
        class_dict[traffic_class] += 1
      else:
        class_dict[f"{traffic_class}"] = 1

    counter = np.array(list(class_dict.values()))

    # Calculate probs
    prior_probs = counter / total_samples

    return prior_probs

prior_probablity_traffic = compute_prior_probabilities_traffic(train_data_traffic)
prior_probablity_traffic

# -> A

array([0.7 , 0.1 , 0.15, 0.05])

In [189]:
# Câu 10, 11, 12 ,13 Xác suất xảy ra sự kiện "Class" = "On Time/Late/ Very Late/ Cancelled" khi sự kiện X xảy ra là:

def compute_conditional_probabilities_traffic(train_data: np.ndarray) -> tuple:
    """
    Calculate conditional probabilities P(Feature|Class) for all features.

    Args:
        train_data: Training dataset

    Returns:
        Tuple of (conditional_probabilities, feature_values)
    """
    class_names = ["On Time", "Late", "Very Late", "Cancelled"]
    n_features = train_data.shape[1] - 1  # Exclude target column
    conditional_probs = []
    feature_values = []

    for feature_idx in range(n_features):
        # Get unique values for this feature -> Theo từng cột, Example: cột thứ 3: Weather -> unique value: sunny, rain,...
        unique_values = np.unique(train_data[:, feature_idx])
        feature_values.append(unique_values)

        # Initialize conditional probability matrix
        feature_cond_probs = np.zeros((len(class_names), len(unique_values)))

        for class_idx, class_name in enumerate(class_names):
            # Get samples for this class
            samples = train_data[train_data[:,-1] == class_name]   # Lấy list những sample cùng No/Yes

            for value_idx, value in enumerate(unique_values):
                # Count occurrences of this feature value in this class
                sub_samples = samples[:,feature_idx] # Lấy cột của feature đang tính
                count = 0
                for sample in sub_samples:
                  if sample == value:
                    count += 1
                # Calculate conditional probability
                if len(samples) != 0:
                    feature_cond_probs[class_idx][value_idx] = count / len(samples)
                else:
                    feature_cond_probs[class_idx][value_idx] = 0
                ### Your code here

        conditional_probs.append(feature_cond_probs)

    return conditional_probs, feature_values

In [190]:
def train_naive_bayes_traffic(train_data):
    """
    Train the Naive Bayes classifier.

    Args:
        train_data: Training dataset

    Returns:
        Tuple of (prior_probabilities, conditional_probabilities, feature_names)
    """

    # Calculate prior probabilities
    prior_probabilities = compute_prior_probabilities_traffic(train_data)

    # Calculate conditional probabilities
    conditional_probabilities, feature_names = compute_conditional_probabilities_traffic(train_data)

    return prior_probabilities, conditional_probabilities, feature_names

In [191]:
# Train the model
prior_probs, conditional_probs, feature_names = train_naive_bayes_traffic(train_data_traffic)

In [192]:
def predict_traffic(
        X, prior_probabilities, conditional_probabilities, feature_names
    ):
    """
    Make a prediction for given features.

    Args:
        X: List of feature values
        prior_probabilities: Prior probabilities for each class
        conditional_probabilities: Conditional probabilities for each feature
        feature_names: Names/values for each feature

    Returns:
        Tuple of (prediction, probabilities)
    """
    class_names = ["On Time", "Late", "Very Late", "Cancelled"]

    # Get feature indices
    feature_indices = []
    for i, feature_value in enumerate(X):
        feature_indices.append(get_feature_index(feature_value, feature_names[i]))

    # Calculate probabilities for each class
    class_probabilities = []

    for class_idx in range(len(class_names)):
        # Start with prior probability
        prior_prob = prior_probabilities[class_idx]

        class_probabilities.append(prior_prob)
        # Multiply by conditional probabilities
        for i in range(len(conditional_probabilities)):
          conditional_prob = conditional_probabilities[i][class_idx][feature_indices[i]]
          class_probabilities[class_idx] *= conditional_prob

    # Make prediction
    predicted_class_idx = np.argmax(class_probabilities)
    prediction = class_names[predicted_class_idx]

    # Create probability dictionary
    prob_dict = {
    class_name: round(prob.item(), 4)
    for class_name, prob in zip(class_names, class_probabilities)
    }

    return prediction, prob_dict

In [193]:
X = ["Weekday", "Winter", "High", "Heavy"]

prediction, prob_dict = predict_traffic(
    X, prior_probs, conditional_probs, feature_names
)

prediction, prob_dict


# Câu 10 C
# Câu 11: D
# Câu 12 A
# Câu 13: D
# Câu 14: C

('Very Late',
 {'On Time': 0.0026, 'Late': 0.0, 'Very Late': 0.0222, 'Cancelled': 0.0})

### III.3. Iris Classification


In [204]:
data = np.array([
    [1.4, 0],
    [1.0, 0],
    [1.3, 0],
    [1.9, 0],
    [2.0, 0],
    [1.8, 0],
    [3.0, 1],
    [3.8, 1],
    [4.1, 1],
    [3.9, 1],
    [4.2, 1],
    [3.4, 1]
])

In [208]:
from sklearn.naive_bayes import GaussianNB


X_train = data[:,0].reshape((-1, 1))
y_train = data[:,-1]

In [213]:
gauss_nb = GaussianNB()

gauss_nb.fit(X_train, y_train)


# Mean
print(gauss_nb.theta_)

# Variance
print(gauss_nb.var_)

#Câu 15: A
# Câu 16: B


[[1.56666667]
 [3.73333333]]
[[0.12888889]
 [0.17222222]]


In [217]:
# Câu 17 Xác suất dữ liệu kiểm thử X thuộc vào "Class" = "0" và "Class" = "1" lần lượt là:
X_test = np.array([[3.4]])

print(gauss_nb.predict_proba(X_test))


# Log của P(X,C)
log_probs = gauss_nb.predict_joint_log_proba(X_test)

# chuyển log -> số thường
raw_probs = np.exp(log_probs)

for cls, prob in zip(gauss_nb.classes_, raw_probs[0]):
    print(f'P(Class={cls}|X) ∝ {prob:.6f}')

# Câu

[[3.47019998e-06 9.99996530e-01]]
P(Class=0.0|X) ∝ 0.000001
P(Class=1.0|X) ∝ 0.348129
